# Clase 1: Inferencia Bayesiana

En el caso de estar usando Google Colab, necesitamos instalar PreliZ y AviZ-base.

In [ ]:
%%capture
# En Colab instalamos PreliZ. En un entorno que ya lo tiene no hace falta.
import importlib.util
if importlib.util.find_spec("preliz") is None or importlib.util.find_spec("arviz_base") is None:
    %pip install preliz
    %pip install arviz_base

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
import preliz as pz

COLORES = ["#0072B2", "#E69F00", "#009E73", "#CC79A7"]

# Regla de Bayes

La regla de Bayes no es más que una consecuencia directa de la regla del producto:

$$
\begin{aligned}
P(A, B) &= P(A \mid B) P(B) \\
P(A, B) &= P(B \mid A) P(A)
\end{aligned}
$$

Como ambos términos a la derecha de la igualdad son iguales, podemos escribir:

$$
P(A \mid B) P(B) = P(B \mid A) P(A)
$$

y reordenando llegamos a

$$
P(A \mid B) = \frac{P(B \mid A)P(A)}{P(B)}
$$

que es la famosa regla de Bayes.

# Modelos bayesianos

Escribir la regla de Bayes de la siguiente manera no dice mucho:

$$
P(A \mid B) = \frac{P(B \mid A)P(A)}{P(B)}
$$

Además, lo único bayesiano en ella es el nombre.

Hay dos ideas centrales que hacen que un método sea **bayesiano**:

* Toda **cantidad desconocida** es modelada utilizando una **distribución de probabilidad** de algún tipo.
* La regla de Bayes es usada para **actualizar dicha distribución a la luz de los datos**.

En el universo bayesiano, las cantidades conocidas son consideradas fijas y usualmente les llamamos datos ($y$ o $\boldsymbol{y}$).

Por el contrario, las cantidades desconocidas se consideran variables aleatorias y conforman el conjunto de parámetros ($\theta$ o $\boldsymbol{\theta}$).

Para construir un modelo bayesiano necesitamos:

* Un **modelo para las observaciones**, que nos habla de cómo se pudieron haber generado los datos $p(Y = y\mid \theta)$ (a.k.a. **verosimilitud** o **_likelihood_**).
* Un **modelo, _a priori_,** para los **parámetros** del modelo observacional $p(\theta)$ (a.k.a. **_prior_**).

La combinación de ambos da lugar a lo que se conoce como _full Bayesian model_:

$$
p(y, \theta) = p(y \mid \theta) p(\theta)
$$

que refleja **todos nuestros supuestos** sobre el fenómeno bajo estudio.

Dado un conjunto de observaciones, se obtiene una distribución _a posteriori_ para los parámetros del modelo:

$$
p(\theta \mid Y = y)
    = \frac{p(Y = y \mid \theta) p(\theta)}{p(Y = y)}
    =  \frac{\overbrace{p(Y = y\mid \theta)}^{\text{likelihood}} \overbrace{p(\theta)}^{\text{prior}}}{\underbrace{\int_{\Theta} p(Y =y \mid \theta) p(\theta) \text{d}\theta}_{\text{likelihood marginal}}}
$$

De manera más concisa, se suele escribir:

$$
p(\theta \mid y) = \frac{p(y \mid \theta) p(\theta)}{p(y)}
$$

El denominador, llamado _likelihood marginal_ o evidencia, es el _likelihood_ promediado sobre todas las posibles hipótesis $\theta$ (i.e. configuraciones posibles del modelo observacional).

El _likelihood marginal_ puede ser visto como una constante de normalización que hace que la densidad _a posteriori_ integre a 1. Por esa razón también se suele ver:

$$
p(\theta \mid Y = y) \propto p(Y = y\mid \theta)p(\theta) = p(Y = y, \theta)
$$

El trabajo de un bayesiano no termina en el hallazgo de la distribución _a posteriori_ de $\theta$ expresado como una función de densidad.

A partir de esa distribución _a posteriori_ se obtienen cantidades de interés:

* Media
* Varianza
* Percentiles
    * Mediana
    * Intervalos de credibilidad
* Probabilidades

¿Qué tienen en común estas cantidades? Todas se obtienen como valores esperados de la forma:

$$
\mathbb{E}[\varphi(\theta)] = \int \varphi(\theta) p(\theta \mid Y = y) d\theta
$$

Por ejemplo, la media o esperanza _a posteriori_ de $\theta$ es

$$
\mathbb{E}[\theta] = \int \theta p(\theta \mid Y = y) d\theta
$$

## Resumen

Un enfoque bayesiano es aquel que utiliza distribuciones de probabilidad para cuantificar la incertidumbre.

Se necesita:

1. Asumir un modelo observacional $p(Y = y \mid \theta)$.
2. Asumir un modelo para los parámetros de configuración del modelo observacional $p(\theta)$.
3. Actualizar la distribución sobre los parámetros a la luz de los datos, obteniendo $p(\theta \mid Y = y)$.
4. Extraer _insights_ de interés en forma de valores esperados obtenidos a partir del _posterior_.

## Una dosis de realidad

Dados nuestros supuestos, lo mejor que podemos hacer para actualizar nuestro conocimiento a partir de los datos es utilizar la regla de Bayes.

Sin embargo, no debemos perder de vista que la fiabilidad de los resultados que obtengamos va a depender de qué tan compatibles son nuestros supuestos iniciales con el "verdadero" proceso generador de datos. Es decir, es necesario evaluar el modelo. Algunas posibilidades son:

* Comparar las predicciones generadas por el modelo con los datos observados
* Evaluar si las conclusiones obtenidas tienen sentido dado el marco conceptual en  que se trabaja
* Determinar si los resultados obtenidos son sensibles a detalles del modelo (sobre todo aquellos detalles para los cuales no tenemos demasiada información)
* ...

Además, es posible que realizar inferencia bayesiana sea demasiado costosa en la práctica por lo que sea conveniente incorporar simplificaciones.

A fin de cuentas, el desarrollo de un modelo bayesiano es casi siempre el resultado de un proceso iterativo donde uno va logrando, paso a paso, un mayor nivel de granularidad en la descripción de un fenómeno, hasta llegar a un nivel que sea "lo suficientemente bueno".

# Problemas

En esta sección vamos a trabajar con tres problemas desde una óptica bayesiana de manera analítica, sin utilizar PyMC. El objetivo es repasar los componentes fundamentales del modelado bayesiano y luego motivar el uso de la simulación computacional para realizar inferencias. En las siguientes _notebooks_, usaremos PyMC para trabajar con modelos más complejos.

## Agua en la Tierra

¿Qué proporción de la superficie de la Tierra está cubierta por agua?

Siguiendo el ejemplo de McElreath en *Statistical Rethinking* (capítulo 2), hacemos girar un globo, señalamos un punto y registramos si encontramos agua o tierra. Repetimos el experimento 9 veces y observamos 6 veces agua y 3 veces tierra.

A partir de los datos, ¿qué se puede concluir sobre la proporción de agua en la Tierra?

¿Qué supuesto(s) realizamos? ¿Cuál(es)?

A pensar un poco...

**¿Cuál es un modelo observacional plausible?**

La variable observada es la **cantidad de veces que observamos agua**. Es un conteo. Toma valores enteros y su mínimo valor posible es 0 (cuando no observamos agua en ningún ensayo). El máximo valor que puede tomar es la cantidad de veces que hacemos girar el globo (cuando observamos agua en todos los ensayos).

Es posible pensar que cada giro del globo es un intento o ensayo con dos resultados posibles: éxito (cuando observamos agua) y fracaso (cuando observamos tierra). Este tipo de ensayo, con dos resultados posibles, se conoce como [ensayo de Bernoulli](https://es.wikipedia.org/wiki/Ensayo_de_Bernoulli). La cantidad observada es la cantidad de éxitos obtenidos.

También se puede suponer que en cada ensayo hay una probabilidad $\theta$ de que el punto señalado esté cubierto por agua.
Si se asume que esta probabilidad de éxito es constante y que los resultados de los ensayos son todos independientes entre sí dado $\theta$, la sucesión de ensayos (cada vez que hacemos girar el globo) se conoce como [proceso de Bernoulli](https://es.wikipedia.org/wiki/Proceso_de_Bernoulli).

Bajo los supuestos mencionados, la cantidad de éxitos obtenidos en una cantidad fija de ensayos sigue una [distribución binomial](https://es.wikipedia.org/wiki/Distribuci%C3%B3n_binomial). En otras palabras, si estamos dispuestos a asumir los supuestos de los párrafos anteriores, surge naturalmente la distribución binomial como modelo observacional para describir el número de veces que observamos agua en $N$ intentos.

Escribimos

$$
Y \mid \theta \sim \text{Binomial}(N, \theta)
$$

donde:

* $Y$ cantidad de éxitos en $N$ ensayos (cantidad de veces que observamos agua en $N$ giros).
* $\theta$ probabilidad de obtener un éxito en un ensayo cualquiera (proporción de agua en la superficie de la Tierra).

Para identificar esa probabilidad con la proporción de agua, estamos suponiendo que el mecanismo de selección no favorece ninguna región de la superficie. Si siempre señalamos cerca del ecuador, por ejemplo, ¿estaríamos respondiendo la misma pregunta?

Veamos qué resultados podríamos observar para distintos valores de $\theta$.

In [ ]:
n_globo, y_globo = 9, 6
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True, layout="constrained")
for ax, probabilidad in zip(axes, [0.25, 0.5, 0.75]):
    valores = np.arange(n_globo + 1)
    modelo = pz.Binomial(n=n_globo, p=probabilidad)
    ax.bar(valores, modelo.pdf(valores), color=COLORES[0])
    ax.set(xlabel="$y$", title=rf"$\theta={probabilidad}$", xticks=[0, 3, 6, 9])
axes[0].set_ylabel("Probabilidad");

**¿Cuáles son las cantidades conocidas y desconocidas de este modelo?**

La distribución binomial tiene dos parámetros (o variables de configuración). La cantidad de ensayos $N$ es una cantidad fija y conocida que es igual a $N = 9$ en nuestro problema. Por otro lado, la probabilidad de éxito $\theta$ es una cantidad desconocida. De hecho, es la cantidad para la cuál queremos hacer inferencia.

Una vez observados los datos, fijamos $y=6$ y variamos $\theta$:

$$
p(Y=6\mid\theta)=\binom{9}{6}\theta^6(1-\theta)^3.
$$

Esta es la **función de verosimilitud**. Nos permite comparar valores de $\theta$ según su compatibilidad con los datos observados. ¡No es una distribución de probabilidad sobre $\theta$!

In [ ]:
theta_grid = np.linspace(0.0001, 0.9999, 1000)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(theta_grid, pz.Binomial(n=n_globo, p=theta_grid).pdf(y_globo), color=COLORES[1])
ax.set(xlabel=r"$\theta$", ylabel=r"$p(y=6\mid\theta)$");

**¿Cuál es un modelo plausible para las cantidades desconocidas?**

Hay un solo parámetro desconocido en nuestro modelo, $\theta$. Un primer paso para proponer un modelo plausible (i.e. una distribución _a priori_) para este es determinar el dominio o soporte del parámetro.

El parámetro $\theta$ es una probabilidad. Es una cantidad continua que toma valores mayores o iguales a 0 y menores o iguales a 1. Por lo tanto, tenemos que considerar a las distribuciones de probabilidad cuyo soporte es el intervalo $[0, 1]$.

Una de las familias de distribuciones con soporte en el intervalo  $[0, 1]$ más populares es la [distribución beta](https://es.wikipedia.org/wiki/Distribuci%C3%B3n_beta). Es popular por su flexibilidad para representar diferentes supuestos sobre la probabilidad de éxito y su conveniencia computacional en la estadística bayesiana.

En su parametrización más común, la distribución beta depende de dos parámetros de forma positivos $\alpha$ y $\beta$.

Veamos la forma de la función de densidad para diferentes combinaciones de los parámetros $\alpha$ y $\beta$.

In [ ]:
beta_params = [
    (1, 1),
    (10, 3),
    (4, 9),
    (10, 20),
    (30, 70),
    (60, 140),
    (300, 600)
]

fig, axes = plt.subplots(
    2, 4, figsize=(14, 6), sharey=False, layout="constrained"
)
beta_grid = np.linspace(0, 1, 1000)

for ax, (alpha, beta) in zip(axes.flat, beta_params):
    distribucion = pz.Beta(alpha=alpha, beta=beta)
    ax.plot(beta_grid, distribucion.pdf(beta_grid), color=COLORES[0])
    ax.set(
        title=f"Beta({alpha}, {beta})",
        xlabel=r"$\theta$",
        ylabel="Densidad",
        xlim=(0, 1),
        ylim=(0, None),
    )
    ax.tick_params(axis="y", left=True, labelleft=True)

# Hay siete distribuciones y ocho lugares en la grilla.
fig.delaxes(axes.flat[-1]);

Si bien $\alpha$ y $\beta$ son cantidades continuas, es posible interpretarlas como (pseudo-) cantidades de éxitos y fracasos. Manteniendo $\beta$ fijo, cuando $\alpha$ crece, mayor concentración de probabilidad alrededor de valores cercanos a 1. Manteniendo $\alpha$ fijo, cuando $\beta$ crece, mayor concentración de probabilidad en valores cercanos a 0.

Para este primer cálculo usamos un *prior* uniforme en $\theta$: asignamos la misma probabilidad a intervalos de igual longitud dentro de $[0,1]$. La distribución $\text{Beta}(1,1)$ es justamente una $\text{Uniforme}(0,1)$.

No estamos usando información sobre mapas o mediciones previas de la superficie terrestre. Más adelante vamos a ver qué pasa si cambiamos este supuesto.

Luego de una larga explicación de por qué proponemos el modelo que proponemos, podemos presentarlo resumidamente de la siguiente forma:

$$
\begin{aligned}
Y \mid \theta &\sim \text{Binomial}(N, \theta) \\
\theta &\sim \text{Beta}(\alpha, \beta)
\end{aligned}
$$

Este modelo es muy popular en estadística bayesiana y se conoce como modelo beta-binomial. A partir de este modelo bayesiano y los datos observados, simplemente la cantidad de éxitos $y=6$, queremos obtener el _posterior_.

Utilizando la regla de Bayes, es posible demostrar que la densidad _a posteriori_ $p(\theta \mid Y = y)$ es también la de una distribución beta con parámetros:

$$
\begin{aligned}
\alpha_\text{posterior} &= y + \alpha_\text{prior} \\
\beta_\text{posterior} &= N - y + \beta_\text{prior}
\end{aligned}
$$

#### Demostración

El modelo conjugado Beta–Binomial se escribe:

$$
\begin{aligned}
Y \mid \theta &\sim \text{Binomial}(N, \theta) \\
\theta &\sim \text{Beta}(\alpha, \beta)
\end{aligned}
$$

y se cuenta con las siguientes funciones de masa de probabilidad y de densidad:

$$
\begin{aligned}
p(y \mid \theta) &= \binom{N}{y} \theta^y (1-\theta)^{N-y} \\
p(\theta \mid \alpha, \beta) &= \frac{1}{B(\alpha,\beta)} \theta^{\alpha - 1} (1-\theta)^{\beta - 1}
\end{aligned}
$$

donde $B(\alpha,\beta) = \dfrac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha+\beta)}$ es la función Beta.

La densidad _a posteriori_ de $\theta$ es proporcional al producto de $p(y \mid \theta)$ y $p(\theta \mid \alpha, \beta)$:

$$
\begin{aligned}
p(\theta \mid y) &\propto
    \binom{N}{y} \theta^y (1-\theta)^{N-y} \cdot
    \frac{1}{B(\alpha,\beta)} \theta^{\alpha - 1} (1-\theta)^{\beta - 1} \\
&= \displaystyle \frac{\binom{N}{y}}{B(\alpha,\beta)} \;
    \theta^{y+\alpha-1} (1-\theta)^{N-y+\beta-1}.
\end{aligned}
$$

Esto puede expresarse como:

$$
p(\theta \mid y) = K^* \; \theta^{y+\alpha-1} (1-\theta)^{N-y+\beta-1}.
$$

Por condición de cierre de una función de densidad Beta:

$$
\int_0^1 K^* \; \theta^{y+\alpha-1} (1-\theta)^{N-y+\beta-1} \, d\theta = 1,
$$

se tiene que:

$$
K^* = \frac{1}{B(y+\alpha, N-y+\beta)}.
$$

Finalmente:

$$
\theta \mid y \sim \text{Beta}(y+\alpha, \; N-y+\beta).
$$


En nuestro caso tenemos $y = 6$ en $N = 9$ giros. Si utilizamos un _prior_ uniforme, se tiene:

$$
\begin{aligned}
Y \mid \theta &\sim \text{Binomial}(9, \theta) \\
\theta &\sim \text{Beta}(1, 1) \\
\theta \mid y &\sim \text{Beta}(\underbrace{7}_{6 + 1}, \underbrace{4}_{9 - 6 + 1})
\end{aligned}
$$

Se dice que la familia beta es **conjugada** de la binomial: el *prior* y el *posterior* pertenecen a la misma familia de distribuciones.

Y podemos graficar la densidad a _priori_ junto con la densidad _a posteriori_.

In [ ]:
def plot_prior_posterior(prior, posterior, grid, label_prior, label_posterior):
    prior_pdf = prior.pdf(grid)
    posterior_pdf = posterior.pdf(grid)

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(grid, prior_pdf, color=COLORES[0], label=label_prior)
    ax.plot(grid, posterior_pdf, color=COLORES[1], label=label_posterior)
    ax.legend()
    return fig, ax

In [ ]:
def marcar_intervalo(ax, distribucion, color):
    inferior, superior = distribucion.eti(mass=0.95, fmt="none")
    ax.plot([inferior, superior], [0, 0], color=color, linewidth=2.5, clip_on=False)
    ax.plot(distribucion.mean(), 0, "o", color=color, clip_on=False, zorder=5)


def formatear_intervalo(intervalo):
    inferior, superior = intervalo
    return f"[{inferior:.3g}, {superior:.3g}]"


def resumir(distribucion):
    inferior, superior = distribucion.eti(mass=0.95, fmt="none")
    print(f"Media: {distribucion.mean():.3g}")
    print(f"IC central del 95%: [{inferior:.3g}, {superior:.3g}]")

In [ ]:
# Parámetros del prior
a_prior, b_prior = 1, 1

# Parámetros del posterior
a_posterior, b_posterior = 1 + y_globo, 1 + n_globo - y_globo

# Prior y posterior en preliz
theta_prior = pz.Beta(a_prior, b_prior)
theta_posterior = pz.Beta(a_posterior, b_posterior)
theta_grid = np.linspace(0.0001, 0.9999, num=200)

fig, ax = plot_prior_posterior(
    prior=theta_prior,
    posterior=theta_posterior,
    grid=theta_grid,
    label_prior="$p(\\theta)$",
    label_posterior="$p(\\theta \\mid y)$",
)
ax.set(xlabel="$\\theta$", ylabel="Densidad");


Es posible ver la "actualización" que se produjo en el conocimiento sobre $\theta$. _A priori_, pensábamos que todos los valores de $\theta$ eran igualmente probables. _A posteriori_, vemos que los valores más plausibles de $\theta$ son aquellos entre 0.3 y 0.9 (construyendo un intervalo a ojo).

Sin embargo, el gráfico que creamos por sí solo no da respuesta a nada... ¿Cómo resumimos la información en la distribución para responder a la pregunta inicial? Calculando cantidades de interés a partir de la distribución _a posteriori_ (i.e. valores esperados).

En particular, busquemos la media _a posteriori_ y un intervalo de credibilidad del 95%. Esto nos va a dar un rango donde se encuentra la mayor parte de la masa de probabilidad y por lo tanto los valores más plausibles de $\theta$.

Para obtener la media _a posteriori_ simplemente usamos el método `.mean()` de la distribución creada con PreliZ.

In [ ]:
theta_posterior.mean() # es igual a alfa / (alfa + beta)

Y para un intervalo de credibilidad el método `.eti()` (_Equal-tailed interval_).

In [ ]:
print(formatear_intervalo(theta_posterior.eti(mass=0.95, fmt="none")))

Este intervalo es central, de colas iguales: deja un 2.5% de probabilidad en cada cola de la distribución.

De esta manera, podemos dar respuesta a la pregunta inicial (¿qué proporción de la superficie de la Tierra está cubierta por agua?) de dos maneras distintas:

* Si tenemos que responder con un solo número, podemos usar la media y decir aproximadamente 64%.
* Si se nos permite reportar un intervalo, diremos que asignamos un 95% de probabilidad a que la proporción esté entre aproximadamente 35% y 88%.

¡Dados nuestros supuestos y los datos observados! La respuesta completa sigue siendo la distribución _a posteriori_.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(theta_grid, theta_posterior.pdf(theta_grid), color=COLORES[0])
marcar_intervalo(ax, theta_posterior, COLORES[0])
ax.set(xlabel=r"$\theta$", ylabel="Densidad", ylim=(0, None))
resumir(theta_posterior);

### Más datos, mismo mecanismo

¿Podemos mejorar esta respuesta? Parece poco...

Imaginemos que seguimos haciendo girar el globo. Para construir este ejemplo hipotético suponemos una proporción verdadera de agua de $2/3$, pero no usamos ese valor para hacer la inferencia.

Consideremos 12 aguas en 18 ensayos en total y 20 aguas en 30 ensayos en total. Son resultados posibles, no garantizados. Mantenemos la misma proporción observada para ver el efecto de tener más datos. No estamos simulando hasta conseguir esos resultados ni sumando tres muestras independientes.

In [ ]:
def graficar_beta_binomial(a_prior, b_prior, y, n, intervalo=False):
    fig, axes = plt.subplots(1, 3, figsize=(12, 3), layout="constrained")

    a_posterior = a_prior + y
    b_posterior = b_prior + n - y

    theta_grid = np.linspace(0.0001, 0.9999, num=256)

    theta_prior_pdf = pz.Beta(a_prior, b_prior).pdf(theta_grid)
    theta_posterior_pdf = pz.Beta(a_posterior, b_posterior).pdf(theta_grid)
    theta_likelihood = pz.Binomial(n=n, p=theta_grid).pdf(y)

    axes[0].plot(theta_grid, theta_prior_pdf, color=COLORES[0])

    axes[1].plot(theta_grid, theta_likelihood, color=COLORES[1])

    axes[2].plot(theta_grid, theta_posterior_pdf, color=COLORES[2])

    title_prior = f"$p(\\theta) \\equiv$Beta({a_prior}, {b_prior})"
    title_likelihood = f"$p(y \\mid \\theta) = \\binom{{{n}}}{{{y}}}\\theta^{{{y}}}(1 - \\theta)^{{{n - y}}}$"
    title_posterior = f"$p(\\theta \\mid y) \\equiv$Beta({a_posterior}, {b_posterior})"

    axes[0].set(yticks=[], xlabel="$\\theta$", title=title_prior)
    axes[1].set(yticks=[], xlabel="$\\theta$", title=title_likelihood)
    axes[2].set(yticks=[], xlabel="$\\theta$", title=title_posterior)

    if intervalo:
        marcar_intervalo(axes[2], pz.Beta(a_posterior, b_posterior), COLORES[2])
    fig.suptitle(f"{y} éxitos en {n} ensayos")
    return fig, axes

In [ ]:
for y, n in [(12, 18), (20, 30)]:
    graficar_beta_binomial(a_prior=1, b_prior=1, y=y, n=n, intervalo=True)
    print(f"{y} aguas en {n} ensayos:")
    resumir(pz.Beta(alpha=1+y, beta=1+n-y))
    plt.show()

Con la misma proporción observada, el _posterior_ se concentra y su media se acerca a $2/3$. Un intervalo más estrecho expresa menor incertidumbre bajo el modelo.

¿Y si en lugar de conseguir más datos usamos información previa diferente? Mantengamos los 6 resultados de agua en 9 ensayos, pero usemos un *prior* $\text{Beta}(10,10)$, más concentrado alrededor de 0.5.

In [ ]:
prior_informativo_globo = pz.Beta(alpha=10, beta=10)
posterior_informativo_globo = pz.Beta(alpha=10+y_globo, beta=10+n_globo-y_globo)
fig, ax = plot_prior_posterior(prior_informativo_globo, posterior_informativo_globo,
                              theta_grid, "Prior: Beta(10, 10)", "Posterior: Beta(16, 13)")
marcar_intervalo(ax, posterior_informativo_globo, COLORES[1])
ax.set(xlabel=r"$\theta$", ylabel="Densidad", ylim=(0, None))
resumir(posterior_informativo_globo);

### El _posterior_ como compromiso entre el _prior_ y los datos

Podemos interpretar $\alpha$ y $\beta$ como pseudoconteos: al observar los datos sumamos $y$ éxitos y $N-y$ fracasos. No son observaciones reales. Otra convención usa $\alpha-1$ y $\beta-1$, los exponentes de la densidad. Aquí nos interesa la interpretación que surge de la media.

La media _a posteriori_ se puede escribir como un promedio ponderado:

$$
\mathbb{E}(\theta\mid y)=\frac{\alpha+y}{\alpha+\beta+N}
=\underbrace{\frac{\alpha+\beta}{\alpha+\beta+N}}_{\text{peso del prior}}
\frac{\alpha}{\alpha+\beta}
+\underbrace{\frac{N}{\alpha+\beta+N}}_{\text{peso de los datos}}\frac{y}{N}.
$$

Para $N=9$, los priors $\text{Beta}(1,1)$, $\text{Beta}(2,2)$ y $\text{Beta}(10,10)$ pesan aproximadamente 18%, 31% y 69% en esa media. A mayor concentración del *prior*, más datos necesitamos para desplazar la media.

Veamos algunas combinaciones de priors y datos. Estos son ejemplos de la actualización beta-binomial, sin suponer que todos describen el mismo globo.

Uniforme en $\theta$:

In [ ]:
graficar_beta_binomial(a_prior=1, b_prior=1, y=6, n=9);

Levemente informativo (pocos datos):

In [ ]:
graficar_beta_binomial(a_prior=2, b_prior=2, y=3, n=7);

Levemente informativo (muchos datos):

In [ ]:
graficar_beta_binomial(a_prior=2, b_prior=2, y=40, n=55);

Informativo (pocos datos):

In [ ]:
graficar_beta_binomial(a_prior=10, b_prior=3, y=3, n=5);

Informativo (muchos datos):

In [ ]:
graficar_beta_binomial(a_prior=10, b_prior=3, y=30, n=50);

Cuando el _prior_ es un salvavidas:

In [ ]:
graficar_beta_binomial(a_prior=2, b_prior=2, y=1, n=1);

In [ ]:
graficar_beta_binomial(a_prior=2, b_prior=2, y=0, n=3);

La forma de actualizar es siempre la misma, pero las respuestas dependen de los supuestos. Un *prior* concentrado puede ayudar si representa información pertinente, o llevarnos a conclusiones poco adecuadas si no la representa.

Tampoco tener muchos datos corrige un mecanismo de selección que favorezca sistemáticamente ciertas regiones del globo. Vale la pena volver a nuestra dosis de realidad: ¿es adecuado el modelo para la pregunta que queremos responder?

## Un _rookie_ en la NBA

Caleb Wilson es un basquetbolista de los Chicago Bulls en la National Basketball Association (NBA) de los Estados Unidos.

Al 27 de octubre de 2026, había jugado 4 partidos, acumulado aproximadamente 135 minutos y convertido **17 de los 23 tiros libres** que intentó.

A partir de los datos, ¿qué se puede concluir sobre la probabilidad de que Caleb enceste un tiro libre? Si tuviera que lanzar dos tiros nuevos, ¿cuál sería la probabilidad de que acertara ambos?

Usamos los datos de la [NBA](https://www.nba.com/player/1643410/caleb-wilson) con esa fecha de corte.

A pensar un poco... ¿Qué partes del modelo anterior podemos reutilizar? ¿Qué supuestos estamos haciendo ahora?

Cada tiro tiene dos resultados posibles y observamos la cantidad de aciertos. Volvemos a suponer ensayos independientes dado $\theta$, con una misma probabilidad de éxito. Ahora $\theta$ es la probabilidad de encestar un tiro libre.

Esto supone, por ejemplo, que esa probabilidad no cambia por cansancio o entrenamiento durante el período que modelamos. No usamos los minutos ni los partidos como ensayos: nuestros ensayos son los tiros libres.

Con el mismo *prior* uniforme:

$$
\begin{aligned}
Y\mid\theta &\sim\text{Binomial}(23,\theta)\\
\theta &\sim\text{Beta}(1,1)\\
\theta\mid y &\sim\text{Beta}(1+17,1+23-17)=\text{Beta}(18,7).
\end{aligned}
$$

In [ ]:
n_caleb, y_caleb = 23, 17
a_posterior, b_posterior = 1+y_caleb, 1+n_caleb-y_caleb
theta_caleb = pz.Beta(alpha=a_posterior, beta=b_posterior)
fig, ax = plot_prior_posterior(pz.Beta(1, 1), theta_caleb, theta_grid,
                              "Prior", "Posterior")
marcar_intervalo(ax, theta_caleb, COLORES[1])
ax.set(xlabel=r"$\theta$", ylabel="Densidad", ylim=(0, None))
resumir(theta_caleb);

La media _a posteriori_ es 0.72 y el intervalo central del 95% va de aproximadamente 0.533 a 0.874. En porcentajes: 72% y [53.3%, 87.4%]. Todavía tenemos bastante incertidumbre sobre su efectividad.

### Dos tiros nuevos

¿Cuál es la probabilidad de que Caleb enceste ambos tiros?

Para encontrar la probabilidad de que Caleb Wilson enceste dos tiros libres, tenemos que recurrir a la distribución **predictiva _a posteriori_**. Esto es así porque la pregunta refiere a **cantidad de tiros libres encestados** que es nuestra **variable observada** y no al parámetro $\theta$ que habla de la probabilidad de encestar un tiro libre.

La función de masa de la distribución predictiva a _posteriori_ se define:

$$
p(\tilde{y}  \mid  y) = \int p(\tilde{y} \mid \theta) p(\theta \mid y) d\theta
$$

En nuestro modelo esta integral tiene solución analítica y el resultado es la función de masa de probabilidad de una [distribución beta-binomial](https://es.wikipedia.org/wiki/Distribuci%C3%B3n_beta-binomial) de parámetros $a$, $b$ y $m$. Los primeros dos son aquellos de la distribución beta posterior y el último es la cantidad de ensayos nuevos.

En cuanto a la probabilidad de interés, la podemos escribir de la siguiente manera:

$$
P(\tilde{Y} = 2 \mid Y = y, m = 2)
$$

Dado $\theta$, la probabilidad de acertar ambos es $\theta^2$. Como no conocemos $\theta$, promediamos sobre su distribución _a posteriori_:

$$
P(\tilde Y=2\mid y,m=2)=\mathbb{E}(\theta^2\mid y)
=\frac{18\cdot19}{25\cdot26}\approx0.526.
$$

No es lo mismo que fijar $\theta$ en su media y calcular $0.72^2=0.518$. Los tiros son independientes **dado** $\theta$. Al promediar sobre el mismo $\theta$ desconocido quedan dependientes en la predictiva. Aprender que acertó el primer tiro también nos daría información sobre el segundo.

La distribución beta-binomial también está implementada en PreliZ y podemos visualizarla a través de su función de masa de probabilidad.

In [ ]:
# Instanciar distribución beta-binomial
y_posterior_predictive = pz.BetaBinomial(alpha=a_posterior, beta=b_posterior, n=2)

# Crear grila de valores para y_tilde
y_tilde_grid = np.arange(3)

# Obtener la probabilidad de cada valor en la grilla
y_tilde_pmf = y_posterior_predictive.pdf(y_tilde_grid)

# Crear figura
fig, ax = plt.subplots(figsize=(7, 5))

# Graficar función de masa de probabilidad
ax.bar(y_tilde_grid, y_tilde_pmf, color=COLORES[1])

# Etiquetar
ax.set(xlabel="$\\tilde{y}$", ylabel="$p(\\tilde{y} \\mid y)$", xticks=[0, 1, 2]);

In [ ]:
y_tilde_pmf

In [ ]:
probabilidad_dos = y_posterior_predictive.pdf(2)
print(f"Probabilidad de acertar ambos: {probabilidad_dos:.3g}")
print(f"Fijando la probabilidad en la media: {theta_caleb.mean()**2:.3g}")

### De la integral a la beta-binomial

Para $m$ tiros nuevos y un posterior $\text{Beta}(a,b)$, multiplicamos la masa binomial por la densidad beta e integramos:

$$
\begin{aligned}
p(\tilde Y=k\mid y)
&=\int_0^1\binom{m}{k}\theta^k(1-\theta)^{m-k}
\frac{\theta^{a-1}(1-\theta)^{b-1}}{B(a,b)}\,d\theta\\
&=\binom{m}{k}\frac{B(a+k,b+m-k)}{B(a,b)},\qquad k=0,\ldots,m.
\end{aligned}
$$

La integral que queda es justamente la función beta. Así obtenemos la distribución beta-binomial sin tener que aproximar la integral por simulación.

Si llamamos $\mu=a/(a+b)$ y $s=a+b$, su media y su varianza son:

$$
\mathbb{E}(\tilde Y\mid y)=m\mu,\qquad
\operatorname{Var}(\tilde Y\mid y)=m\mu(1-\mu)\frac{s+m}{s+1}.
$$

Para $m>1$, la varianza es mayor que la de una binomial con probabilidad fija $\mu$. La predictiva incorpora tanto la variabilidad de los tiros como nuestra incertidumbre sobre $\theta$.

De este modo, la probabilidad de que Caleb enceste 2 de 2 tiros nuevos es aproximadamente 52.6%.

Esto no significa que la probabilidad de encestar todos los tiros libres siempre va a ser tan elevada, independientemente de la cantidad de tiros libres ejecutados.

Supongamos que en vez de lanzar 2 tiros libres, Caleb lanzara 20. ¿Cuál sería la probabilidad de que acertara más de 15?

In [ ]:
y_posterior_predictive = pz.BetaBinomial(alpha=a_posterior, beta=b_posterior, n=20)
y_tilde_grid = np.arange(21)
y_tilde_pmf = y_posterior_predictive.pdf(y_tilde_grid)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(y_tilde_grid, y_tilde_pmf, color=np.where(y_tilde_grid > 15, COLORES[1], COLORES[0]))
ax.set(xlabel="$\\tilde{y}$", ylabel="$p(\\tilde{y} \\mid y)$", xticks=y_tilde_grid[::2]);

In [ ]:
probabilidad_mas_15 = 1 - y_posterior_predictive.cdf(15)
print(f"P(más de 15 aciertos): {probabilidad_mas_15:.3g}")
print(f"P(20 de 20): {y_posterior_predictive.pdf(20):.3g}")
mu = theta_caleb.mean()
s = a_posterior + b_posterior
print(f"Media predictiva: {y_posterior_predictive.mean():.3g}")
print(f"Varianza predictiva: {y_posterior_predictive.var():.3g}")
print(f"Varianza binomial con probabilidad fija: {20*mu*(1-mu):.3g}")

Sumamos las probabilidades desde 16 hasta 20, porque la pregunta dice **más de 15**:

$$
P(\tilde Y>15\mid y)=\sum_{k=16}^{20}p(\tilde Y=k\mid y)\approx0.366.
$$

La probabilidad de acertar los 20 es mucho menor que la de acertar los dos del ejemplo anterior.

### Rookie contra rookie

Cooper Flagg, de los Dallas Mavericks, fue el *rookie* con más tiros libres convertidos en la temporada regular 2025–2026: **283 aciertos en 342 intentos**.

Referencias: [totales de Flagg](https://www.basketball-reference.com/players/f/flaggco01.html) y [líderes rookies de 2025–2026](https://www.statmuse.com/nba/ask/rookie-nba-leader-in-total-free-throws).

Si Caleb terminara con 342 intentos en total, ¿qué probabilidad tendría de superar esos 283 aciertos?

Primero podemos hacer una pregunta sobre el parámetro:

> ¿Cuál es la probabilidad de que la efectividad de Caleb supere la proporción observada de Flagg?

Tomamos $\theta_0=283/342\approx0.8275$ como referencia histórica fija. No estamos modelando una efectividad desconocida de Flagg. Calculamos $P(\theta>\theta_0\mid y)$ usando la función de distribución acumulada, `.cdf()` de PreliZ.

In [ ]:
aciertos_flagg, intentos_flagg = 283, 342
theta_0 = aciertos_flagg / intentos_flagg
probabilidad_superar_proporcion = 1 - theta_caleb.cdf(theta_0)
print(f"Proporción de referencia: {theta_0:.3g}")
print(f"P(theta > referencia | y): {probabilidad_superar_proporcion:.3g}")

La probabilidad es aproximadamente 10.55%. Pero... esta pregunta es sobre $\theta$. Para comparar cantidades finales de aciertos usamos la distribución predictiva.

Caleb ya lanzó 23 tiros y convirtió 17. Si termina con 342 intentos, le quedan **319 tiros nuevos**. Sea $\tilde Y$ la cantidad de aciertos en esos tiros:

$$
\tilde Y\mid y\sim\text{BetaBinomial}(319,18,7).
$$

Superar los 283 de Flagg requiere:

$$
17+\tilde Y>283\quad\Longleftrightarrow\quad\tilde Y\ge267.
$$

Necesita al menos 267 nuevos aciertos, para terminar con 284. Por lo tanto:

$$
P(17+\tilde Y>283\mid y)=\sum_{k=267}^{319}p(\tilde Y=k\mid y)=1-F(266).
$$

In [ ]:
tiros_restantes = intentos_flagg - n_caleb
minimo_nuevos = aciertos_flagg + 1 - y_caleb
predictiva_temporada = pz.BetaBinomial(alpha=a_posterior, beta=b_posterior, n=tiros_restantes)
probabilidad_superar_conteo = 1 - predictiva_temporada.cdf(minimo_nuevos - 1)
print(f"Intentos restantes: {tiros_restantes}. Aciertos nuevos necesarios: {minimo_nuevos}")
print(f"Probabilidad de superar los 283 aciertos: {100*probabilidad_superar_conteo:.3g}%")

La probabilidad de que Caleb convierta más tiros libres que Flagg el año anterior, si realiza la misma cantidad de disparos, es de aproximadamente **9.64%**.

¡Dados todos los supuestos que hicimos! Estamos fijando la cantidad total de intentos y suponiendo que una misma probabilidad de acierto describe los tiros observados y futuros.

## Una enfermedad rara

Queremos estimar la proporción de personas que tienen una enfermedad. En una muestra aleatoria de 30 personas encontramos **un caso**. Ahora $\theta$ representa la prevalencia.

A pensar un poco... ¿Cuál es un modelo observacional plausible? Si suponemos personas independientes dado $\theta$, una misma prevalencia y que sabemos sin error quién tiene la enfermedad, podemos reutilizar la binomial.

Como primer ejercicio usamos un *prior* $\text{Beta}(1,1)$ **como si no supiéramos nada sobre la prevalencia**. No representa información previa de que la enfermedad es rara:

$$
\begin{aligned}
Y\mid\theta&\sim\text{Binomial}(30,\theta)\\
\theta&\sim\text{Beta}(1,1)\\
\theta\mid y=1&\sim\text{Beta}(2,30).
\end{aligned}
$$

Tenemos una muestra pequeña y muy pocos casos: elegimos este contexto porque las aproximaciones normales clásicas, como el intervalo de **Wald**, pueden funcionar mal. Su intervalo es simétrico alrededor de la proporción observada y puede extenderse fuera de $[0,1]$ o colapsar si no observamos casos.

Esto no es una limitación de todos los métodos frecuentistas. También hay alternativas como Wilson o Clopper–Pearson.

### Comparar intervalos

El intervalo de Wald del 95% es:

$$
\hat\theta\pm1.96\sqrt{\frac{\hat\theta(1-\hat\theta)}{N}},\qquad \hat\theta=\frac{y}{N}.
$$

Veamos qué obtenemos con un caso y qué pasaría si no encontráramos ninguno.

In [ ]:
n_enfermedad = 30


def intervalo_wald(y, n):
    proporcion = y / n
    semiancho = st.norm.ppf(0.975) * np.sqrt(proporcion * (1-proporcion) / n)
    return np.array([proporcion-semiancho, proporcion+semiancho])


for casos in [1, 0]:
    posterior = pz.Beta(alpha=1+casos, beta=1+n_enfermedad-casos)
    print(f"\n{casos} casos en {n_enfermedad} personas")
    print("Wald 95%:", formatear_intervalo(intervalo_wald(casos, n_enfermedad)))
    resumir(posterior)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    grilla = np.linspace(0, 0.3, 1000)
    ax.plot(grilla, posterior.pdf(grilla), color=COLORES[0])
    marcar_intervalo(ax, posterior, COLORES[0])
    ax.set(xlabel=r"$\theta$", ylabel="Densidad", title=f"{casos} casos en 30 personas", ylim=(0, None))
    plt.show()

Con un caso, Wald produce aproximadamente [-0.031, 0.098], incluyendo prevalencias negativas. El posterior es $\text{Beta}(2,30)$, con media 0.0625 e intervalo central del 95% [0.0079, 0.167].

Sin casos, Wald colapsa a [0,0]. Con el mismo *prior* uniforme, el posterior es $\text{Beta}(1,31)$, con media 0.0312 e intervalo [0.000816, 0.112]. **No observar casos no equivale a conocer que la prevalencia es cero.**

Aunque reportamos dos intervalos del 95%, su interpretación es diferente:

* Un **intervalo de confianza** se construye con un procedimiento diseñado para cubrir el parámetro en el 95% de las muestras repetidas. La aproximación de Wald puede fallar con pocos casos.
* Un **intervalo de credibilidad** contiene el 95% de la probabilidad _a posteriori_, condicionada al modelo, al *prior* y a los datos. Esta es una afirmación sobre nuestra incertidumbre una vez observada la muestra.

### Una corrección sencilla: «más cuatro»

Una corrección para Wald consiste en sumar dos casos y dos no casos:

$$
\tilde\theta=\frac{y+2}{N+4},\qquad
I_{+4}=\tilde\theta\pm1.96\sqrt{\frac{\tilde\theta(1-\tilde\theta)}{N+4}}.
$$

Para un nivel del 95%, esta es la aproximación «más cuatro» de [Agresti y Coull (1998), §3](https://doi.org/10.1080/00031305.1998.10480550).

¿Les resulta familiar ese centro? Si usamos un *prior* $\text{Beta}(2,2)$, la media _a posteriori_ es justamente $(y+2)/(N+4)$.

El centro coincide, pero **los extremos no son idénticos**. El intervalo ajustado sigue siendo una aproximación normal. El intervalo beta se obtiene de sus cuantiles.

In [ ]:
for casos in [1, 0]:
    centro = (casos+2) / (n_enfermedad+4)
    semiancho = st.norm.ppf(0.975) * np.sqrt(centro*(1-centro)/(n_enfermedad+4))
    posterior_mas_cuatro = pz.Beta(alpha=casos+2, beta=n_enfermedad-casos+2)
    print(f"\n{casos} casos: centro ajustado = media posterior = {centro:.3g}")
    print("Intervalo normal +4:", formatear_intervalo(np.array([centro-semiancho, centro+semiancho])))
    print("Intervalo beta 95%:", formatear_intervalo(posterior_mas_cuatro.eti(mass=0.95, fmt="none")))

### Incorporar lo que sabemos de la enfermedad

Ahora sí incorporamos información previa: creemos que entre **1 de cada 100** y **1 de cada 10** personas tienen la enfermedad. Asignamos un 95% de probabilidad a ese intervalo:

$$
P(0.01 \le \theta \le 0.10)=0.95.
$$

¿Cuáles son los parámetros de una beta que representa esa información?

Este proceso de traducir información previa a una distribución se llama **elicitación**. Usamos `maxent` de PreliZ para encontrar una beta con esa masa en el intervalo.


In [ ]:
prior_prevalencia = pz.maxent(pz.Beta(), lower=0.01, upper=0.10, mass=0.95, plot=False)
alpha_prevalencia, beta_prevalencia = prior_prevalencia.params
print(f"alpha = {alpha_prevalencia:.3g}, beta = {beta_prevalencia:.3g}")
resumir(prior_prevalencia)

Obtenemos aproximadamente $\text{Beta}(4.10,75.5)$, con una prevalencia esperada de 5.15%.

Con el mismo caso observado en 30 personas, actualizamos de la manera que ya conocemos:

$$
\theta\mid y=1\sim\text{Beta}(\alpha+1,\beta+29).
$$


In [ ]:
posterior_prevalencia = pz.Beta(alpha=alpha_prevalencia+1, beta=beta_prevalencia+29)
grilla_prevalencia = np.linspace(0, 0.18, 1000)
fig, ax = plot_prior_posterior(
    prior_prevalencia,
    posterior_prevalencia,
    grilla_prevalencia,
    "Prior informado",
    "Posterior"
)
marcar_intervalo(ax, posterior_prevalencia, COLORES[1])
ax.set(xlabel=r"$\theta$", ylabel="Densidad", xlim=(0, 0.18), ylim=(0, None))
resumir(posterior_prevalencia)
inferior, superior = posterior_prevalencia.eti(mass=0.95, fmt="none")
print(f"En porcentajes: media {100*posterior_prevalencia.mean():.3g}%. "
      f"IC 95% [{100*inferior:.3g}%, {100*superior:.3g}%]");

La media _a posteriori_ es aproximadamente **4.65%** y el intervalo central del 95% es **[1.56%, 9.29%]**.

Los datos son los mismos que antes. Lo que cambió fue la información previa que incorporamos al modelo. ¿Tiene sentido esa información para la población que estamos estudiando? ¿Qué tan sensibles son las conclusiones a los supuestos que hicimos?

La regla de Bayes nos dice cómo actualizar. Elegir y evaluar un modelo adecuado sigue siendo parte de nuestro trabajo.
